In [1]:
import os
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import time

import rmm
import cudf
import cupy as cp
import cvxpy as cpv
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

sc.settings.verbosity = 3

# Load our data

In [2]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/pseudotime.h5ad"
adata = sc.read_h5ad(fpath)
adata.obs['dataset'] = 'sample'
adata

CPU times: user 325 ms, sys: 2.24 s, total: 2.56 s
Wall time: 6.8 s


AnnData object with n_obs × n_vars = 15867 × 21412
    obs: 'batch', 'phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_counts', 'n_genes', 'n_reads', 'raw_clusters', 'bbknn_clusters', 'harmony_clusters', 'cluster_str', 'barcoded_phase', 'S_score', 'G2M_score', 'dpt_pseudotime', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'G1_pseudotime', 'G1_order', 'G2M_pseudotime', 'G2M_order', 'mean_pseudotime', 'mean_order', 'nnz', 'dataset'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'highly_variable', 'means', 'dispersions',

# Load Reference

In [3]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/processed/sample_1_adata.h5ad"
bdata = sc.read_h5ad(fpath)
bdata

CPU times: user 4.32 s, sys: 29.3 s, total: 33.7 s
Wall time: 39.7 s


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1125041 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [4]:
%%time

agg = 'sum'

rdata = sc.get.aggregate(
    bdata,
    by='cell_type',
    func=agg,
    layer='counts',
)


rdata.X = rdata.layers[agg]
rdata.obs['dataset'] = 'reference'
print(f"{rdata.shape=}")
rdata.obs.head()

rdata.shape=(188, 52164)
CPU times: user 5.9 s, sys: 4.42 s, total: 10.3 s
Wall time: 10.4 s


,cell_type,dataset
B-1 B cell,B-1 B cell,reference
B-1a B cell,B-1a B cell,reference
B-1b B cell,B-1b B cell,reference
B-2 B cell,B-2 B cell,reference
CD1c-positive myeloid dendritic cell,CD1c-positive myeloid dendritic cell,reference


In [5]:
%%time 
rsc.get.anndata_to_GPU(rdata) 
rsc.get.anndata_to_GPU(adata) 

rsc.pp.normalize_total(rdata, target_sum=1e4)
rsc.pp.log1p(rdata)

rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)

rsc.get.anndata_to_CPU(rdata) 
rsc.get.anndata_to_CPU(adata) 

# Step 1: Get intersection of genes
gene_intersection = sorted(set(adata.var_names) & set(rdata.var_names))
print(f"Intersection gene count: {len(gene_intersection)}")

# Step 2: Align both matrices to intersection
adata = adata[: , gene_intersection].copy()
rdata = rdata[: , gene_intersection].copy()

print(f"adata shape: {adata.shape=} {type(adata)=}")
print(f"rdata shape: {rdata.shape=} {type(rdata)=}")


adata.shape=(15867, 21412) type(adata)=<class 'anndata._core.anndata.AnnData'>
rdata.shape=(188, 52164) type(rdata)=<class 'anndata._core.anndata.AnnData'>
CPU times: user 402 ms, sys: 575 ms, total: 977 ms
Wall time: 1.65 s
